In [ ]:
import requests
import numpy as np
import time
import os
from math import radians, sin, cos, sqrt, atan2
from dotenv import load_dotenv
import json

load_dotenv()

class TrafficScoreCalculator:
    def __init__(self):
        # API Keys
        self.here_api_key = os.getenv('HERE_API_KEY')
        self.openweather_api_key = os.getenv('OPEN_WEATHER_API_KEY')
        self.ticketmaster_api_key = os.getenv('TICKET_MASTER_API_KEY')
        self.google_maps_api_key = os.getenv('GOOGLE_MAPS_API_KEY')
        
        self.poi_categories = {
            'commercial': ['shop', 'office', 'commercial'],
            'retail': ['supermarket', 'mall', 'convenience', 'department_store'],
            'food': ['restaurant', 'cafe', 'fast_food', 'bar', 'pub'],
            'education': ['school', 'university', 'college', 'kindergarten'],
            'healthcare': ['hospital', 'clinic', 'pharmacy', 'doctors'],
            'transport': ['bus_station', 'train_station', 'subway_entrance', 'taxi'],
            'entertainment': ['cinema', 'theatre', 'arts_centre', 'nightclub'],
            'public': ['library', 'post_office', 'courthouse', 'townhall']
        }
        
        # Country population density data (people per km²)
        self.country_densities = {
            'US': 36, 'CA': 4, 'UK': 281, 'GB': 281, 'DE': 232, 'FR': 119,
            'CN': 153, 'IN': 464, 'JP': 347, 'BR': 25, 'RU': 9,
            'AU': 3, 'MX': 66, 'ZA': 49, 'NG': 226, 'EG': 103,
            'IT': 206, 'ES': 94, 'NL': 508, 'BE': 383, 'SE': 25,
            'NO': 15, 'FI': 18, 'DK': 137, 'PL': 124, 'TR': 110,
            'KR': 527, 'ID': 151, 'PK': 287, 'BD': 1265, 'PH': 368
        }
        
    def calculate_distance(self, lat1, lon1, lat2, lon2):
        """Calculate distance between two coordinates in km"""
        R = 6371  # Earth radius in km
        
        lat1_rad = radians(lat1)
        lon1_rad = radians(lon1)
        lat2_rad = radians(lat2)
        lon2_rad = radians(lon2)
        
        dlon = lon2_rad - lon1_rad
        dlat = lat2_rad - lat1_rad
        
        a = sin(dlat/2)**2 + cos(lat1_rad) * cos(lat2_rad) * sin(dlon/2)**2
        c = 2 * atan2(sqrt(a), sqrt(1-a))
        
        return R * c
    
    def get_country_code(self, lat, lon):
        """Get country code from coordinates using Nominatim with proper headers"""
        url = "https://nominatim.openstreetmap.org/reverse"
        headers = {
            'User-Agent': 'TrafficScoreCalculator/1.0 (https://example.com; contact@example.com)'
        }
        params = {
            'format': 'json',
            'lat': lat,
            'lon': lon,
            'zoom': 3,
            'addressdetails': 1
        }
        
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            if 'address' in data and 'country_code' in data['address']:
                return data['address']['country_code'].upper()
            else:
                return 'US'  # Default fallback
                
        except Exception as e:
            print(f"Error getting country code: {e}")
            return 'US'  # Default fallback
    
    def get_real_time_traffic_here(self, lat, lon, radius_km):
        """Get real-time traffic data from HERE API"""
        if not self.here_api_key:
            return 0.7
            
        url = "https://traffic.ls.hereapi.com/traffic/6.2/flow.json"
        params = {
            'apiKey': self.here_api_key,
            'prox': f'{lat},{lon},{radius_km*1000}',
            'responseattributes': 'sh,fc'
        }
        
        try:
            response = requests.get(url, params=params, timeout=10)
            data = response.json()
            
            total_jf = 0
            count = 0
            
            if 'RWS' in data and data['RWS']:
                for rw in data['RWS'][0].get('RW', []):
                    for fis in rw.get('FIS', []):
                        for cf in fis.get('TMC', {}).get('CF', []):
                            jf = cf.get('JF', 0)
                            if jf > 0:
                                total_jf += jf
                                count += 1
            
            return total_jf / count if count > 0 else 0.7
            
        except Exception as e:
            print(f"HERE Traffic API error: {e}")
            return 0.7
    
    def get_weather_impact(self, lat, lon):
        """Get weather impact factor from OpenWeather API"""
        if not self.openweather_api_key:
            return 1.0
            
        url = "https://api.openweathermap.org/data/2.5/weather"
        params = {
            'lat': lat,
            'lon': lon,
            'appid': self.openweather_api_key,
            'units': 'metric'
        }
        
        try:
            response = requests.get(url, params=params, timeout=10)
            data = response.json()
            
            weather_main = data.get('weather', [{}])[0].get('main', '').lower()
            temp = data.get('main', {}).get('temp', 20)
            wind_speed = data.get('wind', {}).get('speed', 0)
            
            # Weather impact factors
            impact = 1.0
            
            # Bad weather increases traffic (people drive instead of walk)
            if any(w in weather_main for w in ['rain', 'snow', 'storm', 'thunderstorm']):
                impact = 1.4
            elif any(w in weather_main for w in ['fog', 'haze', 'mist']):
                impact = 1.2
            elif 'clear' in weather_main or temp > 25:  # Good weather encourages walking
                impact = 0.9
                
            # High winds can reduce outdoor activities
            if wind_speed > 15:
                impact *= 0.9
                
            return impact
            
        except Exception as e:
            print(f"Weather API error: {e}")
            return 1.0
    
    def get_events_impact(self, lat, lon, radius_km):
        """Get events impact from Ticketmaster API"""
        if not self.ticketmaster_api_key:
            return 1.0
            
        url = "https://app.ticketmaster.com/discovery/v2/events.json"
        params = {
            'apikey': self.ticketmaster_api_key,
            'latlong': f'{lat},{lon}',
            'radius': radius_km,
            'unit': 'km',
            'size': 10
        }
        
        try:
            response = requests.get(url, params=params, timeout=10)
            data = response.json()
            
            events_count = len(data.get('_embedded', {}).get('events', []))
            
            # Each event increases traffic potential
            events_impact = 1.0 + (events_count * 0.15)
            return min(events_impact, 2.0)
            
        except Exception as e:
            print(f"Ticketmaster API error: {e}")
            return 1.0
    
    def get_google_places_density(self, lat, lon, radius_km):
        """Get POI density from Google Places API"""
        if not self.google_maps_api_key:
            return 50
            
        url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
        params = {
            'key': self.google_maps_api_key,
            'location': f'{lat},{lon}',
            'radius': min(radius_km * 1000, 50000),
            'type': 'establishment'
        }
        
        try:
            response = requests.get(url, params=params, timeout=10)
            data = response.json()
            
            if data.get('status') == 'OK':
                return len(data.get('results', []))
            else:
                return 50
                
        except Exception as e:
            print(f"Google Places API error: {e}")
            return 50
    
    def get_google_places_breakdown(self, lat, lon, radius_km):
        """Get detailed POI breakdown from Google Places API"""
        if not self.google_maps_api_key:
            return {category: np.random.randint(5, 15) for category in self.poi_categories}
            
        url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
        params = {
            'key': self.google_maps_api_key,
            'location': f'{lat},{lon}',
            'radius': min(radius_km * 1000, 50000)
        }
        
        category_counts = {category: 0 for category in self.poi_categories}
        
        try:
            response = requests.get(url, params=params, timeout=10)
            data = response.json()
            
            if data.get('status') == 'OK':
                for place in data.get('results', []):
                    place_types = place.get('types', [])
                    
                    for category, keywords in self.poi_categories.items():
                        if any(keyword in str(place_types) for keyword in keywords):
                            category_counts[category] += 1
                            
            return category_counts
            
        except Exception as e:
            print(f"Google Places breakdown error: {e}")
            return {category: np.random.randint(5, 15) for category in self.poi_categories}
    
    def query_overpass_roads(self, lat, lon, radius_km):
        """Get road density from Overpass API"""
        radius_m = radius_km * 1000
        radius_deg = radius_m / 111000
        
        min_lat = lat - radius_deg
        max_lat = lat + radius_deg
        min_lon = lon - radius_deg
        max_lon = lon + radius_deg
        
        query = f"""
        [out:json];
        (
          way["highway"]({min_lat},{min_lon},{max_lat},{max_lon});
        );
        out count;
        """
        
        try:
            response = requests.post("https://overpass-api.de/api/interpreter", 
                                   data=query, timeout=15)
            data = response.json()
            
            road_count = 0
            for element in data.get('elements', []):
                if 'tags' in element and 'highway' in element['tags']:
                    road_count += 1
            
            return road_count
            
        except Exception as e:
            print(f"Overpass roads API error: {e}")
            # Estimate based on urban/rural classification
            poi_count = self.get_google_places_density(lat, lon, 2)
            if poi_count > 100:
                return 20  # Urban area
            elif poi_count > 30:
                return 10  # Suburban
            else:
                return 5   # Rural
    
    def get_population_density(self, lat, lon, radius_km):
        """Estimate population density using multiple data sources"""
        country_code = self.get_country_code(lat, lon)
        base_density = self.country_densities.get(country_code, 100)
        
        # Use Google Places density as proxy for urbanization
        poi_density = self.get_google_places_density(lat, lon, 2)
        
        if poi_density > 150:
            # Major urban center
            density_factor = 300
        elif poi_density > 80:
            # Urban area
            density_factor = 150
        elif poi_density > 30:
            # Suburban area
            density_factor = 50
        else:
            # Rural area
            density_factor = 10
            
        estimated_density = base_density * density_factor
        
        # Adjust based on real-time factors
        traffic_flow = self.get_real_time_traffic_here(lat, lon, 1)
        estimated_density *= (0.5 + traffic_flow)  # Scale with traffic
        
        return min(estimated_density, 50000)  # Cap at reasonable maximum
    
    def calculate_traffic_score(self, lat, lon, radius_km=1):
        """
        Calculate comprehensive traffic score using all 5 APIs
        Returns a score between 0-100 with high accuracy
        """
        # Real-time data from all APIs
        real_time_traffic = self.get_real_time_traffic_here(lat, lon, radius_km)
        weather_impact = self.get_weather_impact(lat, lon)
        events_impact = self.get_events_impact(lat, lon, radius_km)
        poi_density = self.get_google_places_density(lat, lon, radius_km)
        population_density = self.get_population_density(lat, lon, radius_km)
        road_density = self.query_overpass_roads(lat, lon, radius_km)
        poi_breakdown = self.get_google_places_breakdown(lat, lon, radius_km)
        
        # Normalize factors (0-1 range)
        norm_traffic = min(1.0, real_time_traffic / 10.0)  # HERE JF is 0-10
        norm_poi = min(1.0, poi_density / 200.0)  # Scale based on maximum expected POIs
        norm_pop = min(1.0, population_density / 20000.0)  # Scale population density
        norm_roads = min(1.0, road_density / 20.0)  # Scale road density
        
        # Calculate base traffic score with optimized weights
        base_score = (
            norm_traffic * 0.30 +      # Real-time traffic flow (most important)
            norm_poi * 0.25 +          # POI density
            norm_pop * 0.20 +          # Population density
            norm_roads * 0.15 +        # Road infrastructure
            (weather_impact - 1.0) * 0.05 +  # Weather impact
            (events_impact - 1.0) * 0.05     # Events impact
        )
        
        # Apply multipliers and ensure 0-100 range
        traffic_score = max(0, min(100, base_score * 100 * weather_impact * events_impact))
        
        # Confidence estimation based on API responses
        confidence_factors = []
        if real_time_traffic > 0: confidence_factors.append(0.9)
        if poi_density > 0: confidence_factors.append(0.85)
        if population_density > 0: confidence_factors.append(0.8)
        
        confidence = np.mean(confidence_factors) if confidence_factors else 0.7
        
        return {
            'traffic_score': round(traffic_score, 1),
            'confidence': round(confidence, 2),
            'real_time_factors': {
                'traffic_flow': round(real_time_traffic, 2),
                'weather_impact': round(weather_impact, 2),
                'events_impact': round(events_impact, 2)
            },
            'infrastructure_factors': {
                'poi_density': poi_density,
                'population_density': round(population_density, 2),
                'road_density': road_density
            },
            'normalized_factors': {
                'traffic': round(norm_traffic, 2),
                'poi': round(norm_poi, 2),
                'population': round(norm_pop, 2),
                'roads': round(norm_roads, 2)
            },
        }

# Example usage with real APIs
def main():
    calculator = TrafficScoreCalculator()
    
    lat = 13.0878
    lon = 80.2785

    result = calculator.calculate_traffic_score(lat, lon, radius_km=2)
    print(json.dumps(result, indent=2))

if __name__ == "__main__":
    main()

{
  "traffic_score": 25.1,
  "confidence": 0.85,
  "real_time_factors": {
    "traffic_flow": 0.7,
    "weather_impact": 0.9,
    "events_impact": 1.0
  },
  "infrastructure_factors": {
    "poi_density": 50,
    "population_density": 27840.0,
    "road_density": 0
  },
  "normalized_factors": {
    "traffic": 0.07,
    "poi": 0.25,
    "population": 1.0,
    "roads": 0.0
  }
}


In [35]:
import geocoder

g = geocoder.ip('me')
print("Latitude:", g.latlng[0])
print("Longitude:", g.latlng[1])


Latitude: 13.0878
Longitude: 80.2785
